# Multi-agent Knowledge Base System Built on Router Pattern

_Building a simple multi-souce knowledge base following router pattern in multi-agent system._

Router decomposes the user query into source-specific sub-questions, routes them to the relevant specialist agents in parallel, and synthesizes results into a coherent answer.

These specialist agents are:

A GitHub Agent: Searches code, issues, and pull requests.

A Notion Agent: Searches internal documentation and wikis.

A Slack Agent: Searches relevant threads and discussions.



In [1]:
# Imports packages

from langchain_ollama import ChatOllama
from typing import Annotated, Literal, TypedDict
from langchain.tools import tool
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
import operator

## Model

_Connecting an appropriate model to a chat client._

In [2]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [3]:
# Initializes chat client
model = ChatOllama(
    model=OLLAMA_MODEL,
    reasoning=False,
    base_url=OLLAMA_ENDPOINT)

## States
Defining states schemas.

In [4]:
class Classification(TypedDict):
    """A single routing decision: which agent to call with what query"""
    source: Literal["github", "notion", "slack"]
    query: str

In [5]:
class AgentInput(TypedDict):
    """Simple input state passed to each sub-agent."""
    query: str

class AgentOutput(TypedDict):
    """Result return by each sub-agent."""
    source: str
    result: str

class RouterState(TypedDict):
    """Main workflow state tracking the query, classifications, results, and final answer."""
    query: str
    classifications: list[Classification]
    results: Annotated[list[AgentOutput], operator.add]     # Uses a reducer `operator.add` to collect outputs from
                                                            # parallel agent executions into a single list.
    final_answer: str


## Tools
Creates tools for each knowledge domain e.g. GitHub, Notion and Slack. 

For this experiment, dummy implementations that return mock data are used. In a production system, these would call actual APIs.

In [6]:
# ========== 3 TOOLS RELATED TO GITHUB ==========
@tool
def search_code(query: str, repo: str = "main") -> str:
    """Searches code in GitHub repositories."""
    return f"Found code matching '{query}' in {repo}: authentication middleware in src/auth.py"

@tool
def search_issues(query: str) -> str:
    """Searches GitHub issues."""
    return f"Found 3 issues matching '{query}': #142 (API auth docs), #89 (OAuth flow), #203 (token refresh)"

@tool
def search_prs(query: str) -> str:
    """Searches pull requests for implementation details."""
    return f"PR #156 added JWT authentication, PR #178 updated OAuth scopes"


# ========== 2 TOOLS RELATED TO NOTION ==========
@tool
def search_notion(query: str) -> str:
    """Searches Notion workspace for documentation."""
    return f"Found documentation: 'API Authentication Guide' - covers OAuth2 flow, API keys, and JWT tokens"


@tool
def get_page(page_id: str) -> str:
    """Get a specific Notion page by ID."""
    return f"Page content: Step-by-step authentication setup instructions"


# ========== 2 TOOLS RELATED TO SLACK ==========
@tool
def search_slack(query: str) -> str:
    """Searches Slack messages and threads."""
    return f"Found discussion in #engineering: 'Use Bearer tokens for API auth, see docs for refresh flow'"

@tool
def get_thread(thread_id: str) -> str:
    """Gets a specific Slack thread."""
    return f"Thread discusses best practices for API key rotation"

## Sub-Agents
_Creates an agent for each vertical. Each agent has domain-specific tools and a prompt optimized for its knowledge source. All three follow the same pattern — only the tools and system prompt differ._

In [7]:
GITHUB_PROMPT=(
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    )

# Creates GitHub sub-agent
github_agent = create_agent(
    model,
    tools=[search_code, search_issues, search_prs],
    system_prompt=GITHUB_PROMPT
)

In [8]:
NOTION_PROMPT=(
        "You are a Notion expert. Answer questions about internal "
        "processes, policies, and team documentation by searching "
        "the organization's Notion workspace."
    )

# Creates Notion sub-agent
notion_agent = create_agent(
    model,
    tools=[search_notion, get_page],
    system_prompt=NOTION_PROMPT
)

In [9]:
SLACK_PROMPT=(
        "You are a Slack expert. Answer questions by searching "
        "relevant threads and discussions where team members have "
        "shared knowledge and solutions."
    )

# Creates Slack sub-agent
slack_agent = create_agent(
    model,
    tools=[search_slack, get_thread],
    system_prompt=SLACK_PROMPT,
)

## Router Workflow Design
The router workflow is built using a `StateGraph`. The main steps in the workflow are as follows:

1. Classify: Analyzes the query and determines which agents to invoke with what sub-questions.
2. Route: Fans out to selected agents in parallel using `Send`.
3. Query Agents: Each agent receives a simple `AgentInput` and returns an `AgentOutput`.
4. Synthesize: Combines collected results into a coherent response.

In [10]:
SYSTEM_PROMPT = """Analyze this query and determine which knowledge bases to consult.
For each relevant source, generate a targeted sub-question optimized for that source.

Available sources:
- github: Code, API references, implementation details, issues, pull requests
- notion: Internal documentation, processes, policies, team wikis
- slack: Team discussions, informal knowledge sharing, recent conversations

Return ONLY the sources that are relevant to the query. Each source should have
a targeted sub-question optimized for that specific knowledge domain.

Example for "How do I authenticate API requests?":
- github: "What authentication code exists? Search for auth middleware, JWT handling"
- notion: "What authentication documentation exists? Look for API auth guides"
(slack omitted because it's not relevant for this technical question)"""

**Classification**

Analyzing the query and determining which agents to invoke with what sub-questions.

In [11]:
# Defines structured output schema for the classifier
class ClassificationResult(BaseModel):
    """Result of classifying a user query into agent-specific sub-questions."""
    classifications: list[Classification] = Field(
        description="List of agents to invoke with their targeted sub-questions"
    )

def classify_query(state: RouterState) -> dict:
    """Classifies query and determine which agents to invoke."""
    
    structured_model = model.with_structured_output(ClassificationResult)

    result = structured_model.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": state["query"]}
    ])

    return {"classifications": result.classifications}

**Routing**

Fanning out to selected agents in parallel using `Send`.

In [12]:
def route_to_agents(state: RouterState) -> list[Send]:
    """Fans out to agents based on classifications."""
    return [
        Send(c["source"], {"query": c["query"]}) for c in state["classifications"]
    ]

**Query Agents**

Each agent receiving a simple `AgentInput` and returing an `AgentOutput.`

In [13]:
def query_github(state: AgentInput) -> dict:
    """Queries the GitHub agent."""
    result = github_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "github", "result": result["messages"][-1].content}]}


def query_notion(state: AgentInput) -> dict:
    """Queries the Notion agent."""
    result = notion_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "notion", "result": result["messages"][-1].content}]}


def query_slack(state: AgentInput) -> dict:
    """Queries the Slack agent."""
    result = slack_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "slack", "result": result["messages"][-1].content}]}

**Synthesizing**

Combining collected results into a coherent response.

In [14]:
SYNTHESIS_PROMPT="""Synthesize these search results to answer the original question: {query}

- Combine information from multiple sources without redundancy
- Highlight the most relevant and actionable information
- Note any discrepancies between sources
- Keep the response concise and well-organized"""

def synthesize_results(state: RouterState) -> dict:
    """Combines results from all agents into a coherent answer."""
    if not state["results"]:
        return {"final_answer": "No results found from any knowledge source."}

    # Format results for synthesis
    formatted = [
        f"**From {r['source'].title()}:**\n{r['result']}" for r in state["results"]
    ]

    synthesis_response = model.invoke([
        {
            "role": "system",
            "content": SYNTHESIS_PROMPT.format(query=state['query'])
        },
        {"role": "user", "content": "\n\n".join(formatted)}
    ])

    return {"final_answer": synthesis_response.content}

**Workflow Compilation**

Assembles the workflow by connecting nodes with edges. 

Using `add_conditional_edges` with the routing function enables parallel execution.

The `add_conditional_edges` call connects the classify node to the agent nodes through the `route_to_agents` function. When `route_to_agents` returns multiple `Send` objects, those nodes execute in parallel.

In [15]:
workflow = (
    StateGraph(RouterState)
    .add_node("classify", classify_query)
    .add_node("github", query_github)
    .add_node("notion", query_notion)
    .add_node("slack", query_slack)
    .add_node("synthesize", synthesize_results)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route_to_agents, ["github", "notion", "slack"])
    .add_edge("github", "synthesize")
    .add_edge("notion", "synthesize")
    .add_edge("slack", "synthesize")
    .add_edge("synthesize", END)
    .compile()
)

## Testing Workflow
Tests the router with queries that span multiple knowledge domains.

In [16]:
result = workflow.invoke({
    "query": "How do I authenticate API requests?"
})

print("Original query:", result["query"])
print("\nClassifications:")
for c in result["classifications"]:
    print(f"  {c['source']}: {c['query']}")
print("\n" + "=" * 60 + "\n")
print("Final Answer:")
print(result["final_answer"])

Original query: How do I authenticate API requests?

Classifications:
  github: Authentication middleware JWT handling
  notion: API Authentication Documentation Guides


Final Answer:
**Authenticating API Requests**

To authenticate API requests, you can use various methods depending on your specific requirements. Here's a comprehensive guide:

### **Using JWT Tokens**

One popular method is to use JSON Web Tokens (JWT) for authentication.

*   **Generate Token**: Your server generates a JWT token using a secret key.
*   **Include Token**: The client includes the JWT token in the `Authorization` header or query parameter of their request.

Here's an example code snippet in Node.js using Express.js and JSON Web Tokens (JWT):

**jwtMiddleware.js**
```javascript
const express = require('express');
const jwt = require('jsonwebtoken');

const secretKey = 'your_secret_key_here';

const authenticate = async (req, res, next) => {
  const token = req.header('Authorization').replace('Bearer ', 